In [1]:
import threading
import time
import random

class RaftNode:
    def __init__(self, node_id, peers):
        self.node_id = node_id
        self.peers = peers
        self.state = 'follower'   # follower, candidate, leader
        self.current_term = 0
        self.voted_for = None
        self.leader_id = None
        self.election_timeout = random.uniform(0.5, 1.5)
        self.last_heartbeat = time.time()
        self.heartbeat_interval = 0.5
        self.running = True

    def start(self):
        self.thread = threading.Thread(target=self.run)
        self.thread.start()

    def run(self):
        while self.running:
            if self.state == 'leader':
                self.send_heartbeats()
                time.sleep(self.heartbeat_interval)
            else:
                if time.time() - self.last_heartbeat > self.election_timeout:
                    self.start_election()
                time.sleep(0.1)

    def send_heartbeats(self):
        print(f"Node {self.node_id} (leader, term {self.current_term}) sends heartbeats")
        for peer in self.peers:
            # In real Raft, we'd send AppendEntries RPC
            peer.receive_heartbeat(self.current_term, self.node_id)

    def receive_heartbeat(self, term, leader_id):
        if term > self.current_term:
            self.current_term = term
            self.state = 'follower'
            self.voted_for = None
        self.leader_id = leader_id
        self.last_heartbeat = time.time()
        if self.state == 'candidate':
            # step down
            self.state = 'follower'

    def start_election(self):
        self.state = 'candidate'
        self.current_term += 1
        self.voted_for = self.node_id
        votes = 1
        print(f"Node {self.node_id} starts election for term {self.current_term}")
        for peer in self.peers:
            if peer.request_vote(self.current_term, self.node_id):
                votes += 1
        if votes > len(self.peers) // 2:
            self.state = 'leader'
            self.leader_id = self.node_id
            print(f"Node {self.node_id} becomes leader for term {self.current_term}")
        else:
            self.state = 'follower'
            self.election_timeout = random.uniform(0.5, 1.5)

    def request_vote(self, term, candidate_id):
        if term > self.current_term:
            self.current_term = term
            self.state = 'follower'
            self.voted_for = None
        if self.voted_for is None and term >= self.current_term:
            self.voted_for = candidate_id
            self.last_heartbeat = time.time()
            return True
        return False

    def stop(self):
        self.running = False
        self.thread.join()

# Example
if __name__ == "__main__":
    nodes = []
    for i in range(3):
        nodes.append(RaftNode(i, []))
    for n in nodes:
        n.peers = [other for other in nodes if other != n]
    for n in nodes:
        n.start()
    time.sleep(10)
    for n in nodes:
        n.stop()

Node 2 starts election for term 1
Node 2 becomes leader for term 1
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
Node 2 (leader, term 1) sends heartbeats
